# Milestone 2 Report
## Predictive Learning Models for the Diabetes Prediction Dataset

**Course:** Machine Learning — Spring 2026  
**Dataset:** [Diabetes Prediction Dataset — Kaggle](https://www.kaggle.com/datasets/iammustafatz/diabetes-prediction-dataset)

---

## Table of Contents

1. [Introduction](#1)
2. [Dataset Description](#2)
3. [Predictive Learning Task](#3)
4. [Existing Kaggle Contributions](#4)
   - 4.1 [Notebook 1 — EDA, SMOTE & Random Forest with GridSearchCV](#4.1)
   - 4.2 [Notebook 2 — XGBoost with PCA for Diabetes & Hypertension](#4.2)
   - 4.3 [Notebook 3 — Multi-Model Comparison on Pima Indians Dataset](#4.3)
5. [Techniques Not Yet Covered in Class](#5)
6. [My Anticipated Contributions](#6)
7. [Conclusion](#7)
8. [References](#8)

---

<a id='1'></a>
## 1. Introduction

Diabetes is a serious chronic condition that affects hundreds of millions of people worldwide and, if left undetected, can lead to complications ranging from cardiovascular disease to kidney failure. Because many of the warning signs show up in routine clinical measurements, machine learning is a natural fit for building early-warning tools that can flag at-risk patients before symptoms become severe.

For this milestone I chose the **Diabetes Prediction Dataset** (Mustafa, 2023) from Kaggle. It combines eight demographic and clinical features from patient health records with a binary diabetes label, making it a clean and meaningful dataset for classification. The dataset is a mix of numeric and categorical variables, It also noted that dataset has class imbalance.. This also means there are genuine data-preprocessing challenges to work through, not just model selection.

This report walks through three Kaggle notebooks that use this dataset, summarises what each author did well, highlights techniques I had not yet encountered in class, and then outlines the novel contributions I plan to make in the mini-project.

---
<a id='2'></a>
## 2. Dataset Description

### 2.1 Overview

The dataset contains **100,000 patient records** stored as a single CSV file. Each row is one patient, described by eight predictor variables and one binary target.

| Feature | Type | Description |
|---|---|---|
| `gender` | Categorical | Male / Female / Other (0.002%, negligible) |
| `age` | Continuous | Patient age in years (range: 0.08–80) |
| `hypertension` | Binary | 1 = diagnosed with hypertension, 0 = not |
| `heart_disease` | Binary | 1 = diagnosed with heart disease, 0 = not |
| `smoking_history` | Categorical | never / No Info / current / former / ever / not current |
| `bmi` | Continuous | Body Mass Index (range: 10.01–95.69) |
| `HbA1c_level` | Continuous | Glycated haemoglobin %; reflects average blood glucose over ~3 months (range: 3.5–9.0) |
| `blood_glucose_level` | Continuous | Fasting blood glucose in mg/dL (range: 80–300) |
| `diabetes` | Binary | **Target** — 1 = diabetic, 0 = not diabetic |

### 2.2 Key Characteristics

A few things stand out when you first look at this dataset:

- **Missing values:** There are none — all 100,000 rows are complete, so no imputation is needed.
- **Duplicate rows:** 3,854 exact duplicates exist and should be dropped before modelling.
- **Class imbalance:** About **91% of patients are non-diabetic (class 0)** and only **9% are diabetic (class 1)**. This is a meaningful imbalance. A model that always predicts "no diabetes" would still hit 91% accuracy, so standard accuracy is not a reliable metric on its own.
- **Mixed variable types:** Two features (`gender`, `smoking_history`) are categorical and need encoding; the remaining six are numeric and can be scaled directly.
- **Medical relevance:** `HbA1c_level` and `blood_glucose_level` are the two markers clinicians actually use to diagnose diabetes (HbA1c ≥ 6.5% or fasting glucose ≥ 126 mg/dL per ADA guidelines), so we would expect them to dominate feature importance.

### 2.3 Descriptive Statistics (raw dataset)

| Statistic | age | bmi | HbA1c_level | blood_glucose_level |
|---|---|---|---|---|
| Mean | 41.9 | 27.3 | 5.53 | 138.1 |
| Std | 22.5 | 6.6 | 1.07 | 40.7 |
| Min | 0.08 | 10.01 | 3.5 | 80 |
| Max | 80 | 95.69 | 9.0 | 300 |

---
<a id='3'></a>
## 3. Predictive Learning Task

The core task here is **supervised binary classification**: given a patient's eight features, predict whether they are diabetic (label = 1) or not (label = 0). More formally, we want to learn a function $f: \mathcal{X} \to \{0, 1\}$ from labelled training pairs $\{(\mathbf{x}_i, y_i)\}_{i=1}^{N}$, where $\mathbf{x}_i \in \mathbb{R}^d$ is the feature vector and $y_i$ is the diagnosis.

### Evaluation Metrics

One thing that makes this task a bit tricky is the class imbalance. Because 91% of patients are non-diabetic, a classifier that always outputs 0 would still achieve 91% accuracy, which is obviously useless. For that reason, accuracy should always be paired with metrics that capture performance on the minority class:

| Metric | Formula | Why it matters here |
|---|---|---|
| Accuracy | $(TP + TN) / N$ | Overall baseline, but misleading alone |
| Precision (class 1) | $TP / (TP + FP)$ | How often a positive prediction is actually correct |
| Recall (class 1) | $TP / (TP + FN)$ | Most important clinically; missing a diabetic patient is costly |
| F1-Score | $2 \cdot P \cdot R / (P + R)$ | Balances precision and recall into one number |
| ROC-AUC | Area under ROC curve | Measures discrimination independent of any single threshold |

---
<a id='4'></a>
## 4. Existing Kaggle Contributions

Three Kaggle notebooks were reviewed for this project. Each one takes a different angle on the same prediction problem, and together they paint a reasonable picture of how the community has approached this dataset. The sections below summarise each author's data preparation choices, modelling decisions, and results.

<a id='4.1'></a>
### 4.1 Notebook 1 — EDA, SMOTE & Random Forest with GridSearchCV

**Author:** @pannmie  
**Kaggle link:** https://www.kaggle.com/code/tumpanjawat/diabetes-eda-random-forest-hp  
**Local reference:** `diabetes-eda-random-forest-hp.ipynb`

#### 4.1.1 Data Preparation

This is the most thorough notebook in terms of exploratory analysis. The author organises the EDA into three levels, moving from single variables up to interactions between multiple features:

- **Univariate analysis:** Histograms and count plots cover age distribution, the heavy skew toward non-smokers, and the target class imbalance.
- **Bivariate analysis:** Boxplots comparing `BMI`, `age`, `HbA1c_level`, and `blood_glucose_level` against the diabetes label show clear separation, particularly for the two clinical markers.
- **Multivariate analysis:** Pair plots and violin plots broken down by gender suggest that diabetic patients tend to be older and carry higher BMI on average.

After the EDA phase, several preprocessing decisions are made before any model is trained:

1. **Duplicate removal:** 3,854 exact duplicate rows are dropped.
2. **Rare-value removal:** The three records where `gender == 'Other'` are excluded, since they represent only 0.002% of the data.
3. **Smoking history recategorisation:** The six original smoking categories are collapsed into three: `non-smoker` (never + No Info), `current`, and `past_smoker` (ever + former + not current). This cuts down cardinality and avoids producing very sparse one-hot columns.
4. **Preprocessing pipeline:** `StandardScaler` is applied to the six numeric features, and `OneHotEncoder` handles `gender` and `smoking_history`, all assembled inside a `ColumnTransformer`.

#### 4.1.2 Handling Class Imbalance

To tackle the 91%/9% split, the author chains SMOTE oversampling with `RandomUnderSampler` inside an `imbalanced-learn` pipeline:

In [ ]:
# Adapted from: diabetes-eda-random-forest-hp.ipynb
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as imbPipeline
from sklearn.ensemble import RandomForestClassifier

over  = SMOTE(sampling_strategy=0.1)               # minority -> 10% of majority
under = RandomUnderSampler(sampling_strategy=0.5)  # then undersample majority

clf = imbPipeline(steps=[
    ('preprocessor', preprocessor),  # ColumnTransformer: StandardScaler + OHE
    ('over',         over),
    ('under',        under),
    ('classifier',   RandomForestClassifier())
])

#### 4.1.3 Model & Hyperparameter Tuning

A `RandomForestClassifier` is tuned via `GridSearchCV` with 5-fold cross-validation over the following grid:

| Hyperparameter | Values searched |
|---|---|
| `n_estimators` | 50, 100, 200 |
| `max_depth` | None, 10, 20 |
| `min_samples_split` | 2, 5, 10 |
| `min_samples_leaf` | 1, 2, 4 |

**Best parameters found:** `max_depth=10`, `min_samples_leaf=1`, `min_samples_split=5`, `n_estimators=100`.

A `max_depth` of 10 prevents individual trees from memorising noise; constraining `min_samples_split` and `min_samples_leaf` further regularises leaf nodes.

#### 4.1.4 Results

| Metric | Class 0 (no diabetes) | Class 1 (diabetes) |
|---|---|---|
| Precision | 0.98 | 0.65 |
| Recall | 0.96 | 0.80 |
| F1-Score | 0.97 | 0.72 |
| **Overall Accuracy** | **95.1%** | |

The model predicts non-diabetic cases with very high reliability (98% precision). Recall for diabetic cases reaches 80%, meaning roughly 1 in 5 true diabetics is missed, which is an important medical limitation.

#### 4.1.5 Feature Importance

Random Forest's built-in Gini impurity importance reveals a clear hierarchy:

| Feature | Importance |
|---|---|
| HbA1c_level | 0.44 |
| blood_glucose_level | 0.32 |
| age | 0.14 |
| bmi | 0.06 |
| hypertension | 0.02 |
| heart_disease | 0.01 |
| smoking_history (all) | ~0.00 |
| gender (all) | ~0.00 |

The two direct clinical markers (`HbA1c_level`, `blood_glucose_level`) together account for 76% of feature importance, which is fully consistent with established medical understanding of diabetes diagnosis.

<a id='4.2'></a>
### 4.2 Notebook 2 — XGBoost with PCA for Diabetes & Hypertension

**Author:** Muhammad Danish Mubashar  
**Kaggle link:** https://www.kaggle.com/code/muhammaddanishmubashar/diabetes-hypertension-predict-acc-97  
**Local reference:** `diabetes-hypertension-predict-acc-97.ipynb`

#### 4.2.1 Data Preparation

The EDA here is more applied than in Notebook 1. The author uses grouped bar plots by gender, hypertension, heart disease, and smoking history, and also breaks down age-range statistics by diabetes status. One interesting observation is that the youngest diabetic patient in the dataset is only 3 years old, while the oldest is 80.

Preprocessing follows four steps:

1. **Duplicate removal:** 3,854 exact duplicates are dropped, consistent with what Notebook 1 found.
2. **Label encoding:** `gender` and `smoking_history` are integer-encoded via `LabelEncoder`. This is simpler than one-hot encoding but does impose an implicit ordinal relationship between categories.
3. **Feature scaling:** `StandardScaler` normalises all eight features before PCA is applied.
4. **PCA:** A cumulative explained-variance plot confirms that all eight principal components are needed to capture 100% of the variance. Since no components are discarded, PCA here is acting as a feature decorrelation step rather than an actual dimensionality reduction.

#### 4.2.2 Model — XGBoost Classifier

In [ ]:
# Adapted from: diabetes-hypertension-predict-acc-97.ipynb
from xgboost import XGBClassifier
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Scale and decorrelate with PCA
scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca   = PCA(n_components=8)    # retains all 8 components (no reduction)
X_pca = pca.fit_transform(X_scaled)

X_train, X_test, y_train, y_test = train_test_split(
    X_pca, y, test_size=0.2, random_state=42)

xgb_model = XGBClassifier(random_state=42)
xgb_model.fit(X_train, y_train)

#### 4.2.3 Results — Diabetes Prediction

| Metric | Class 0 (no diabetes) | Class 1 (diabetes) |
|---|---|---|
| Precision | 0.97 | 0.93 |
| Recall | 0.99 | 0.67 |
| F1-Score | 0.98 | 0.78 |
| **Overall Accuracy** | **96.6%** | |

XGBoost achieves higher precision on diabetic cases (0.93) compared to Notebook 1's Random Forest (0.65), but lower recall (0.67 vs. 0.80). In other words, XGBoost is more conservative: it raises fewer false positives but misses more true diabetics.

#### 4.2.4 Dual-Task Extension — Hypertension Prediction

Uniquely, this notebook reuses the same eight features to train a second XGBoost classifier to predict `hypertension` as the target, achieving **97% accuracy**. This highlights how the same clinical feature set encodes multiple comorbid conditions. Both models are serialised to disk using Python's `pickle` module for later deployment.

<a id='4.3'></a>
### 4.3 Notebook 3 — Multi-Model Comparison on the Pima Indians Dataset

**Author:** Zabihullah  
**Kaggle link:** https://www.kaggle.com/code/zabihullah18/diabetes-prediction  
**Local reference:** `diabetes-prediction.ipynb`

#### 4.3.1 Dataset Note

This notebook does not use the main 100k-record dataset. Instead, it works with the **Pima Indians Diabetes Database** (768 records, 9 features), which is an older and much smaller benchmark. I included it here because it is the only one of the three notebooks that compares multiple classifiers side by side, making it a useful methodological reference even if the dataset differs.

The Pima features are: Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age, and the binary Outcome label.

#### 4.3.2 Data Preparation

1. **EDA:** Histograms, boxplots, pairplots, and a correlation heatmap are produced. The features most correlated with the outcome are Glucose (0.47), BMI (0.29), Age (0.24), and Pregnancies (0.22).
2. **Outlier handling:** IQR-based capping is applied to `Insulin` and `DiabetesPedigreeFunction`. Rather than removing outlier rows, extreme values are simply clipped to the fence limits (Q1 - 1.5×IQR and Q3 + 1.5×IQR).
3. **Train/test split:** An 80/20 random split is used without feature scaling. This is fine for tree-based methods but puts SVM and KNN at a slight disadvantage since both are sensitive to feature magnitudes.

#### 4.3.3 Model Comparison

Eight classifiers are evaluated before and after hyperparameter tuning with `GridSearchCV`:

| Model | Val. Accuracy (before tuning) | Val. Accuracy (after tuning) |
|---|---|---|
| Logistic Regression | 77.7% | — |
| Random Forest | 78.1% | **83.1%** |
| Decision Tree | 76.6% | **86.4%** |
| K-Nearest Neighbours | 66.0% | 72.7% |
| SVM (linear kernel) | 77.2% | — |
| AdaBoost | 74.0% | — |
| Gradient Boosting | 75.3% | — |
| XGBoost | 72.7% | — |

The **tuned Decision Tree** turns out to be the best model at 86.4% validation accuracy. This is somewhat surprising given that Random Forest and gradient boosting methods are generally considered stronger. The untuned Random Forest suffers severe overfitting (100% training vs. 78% validation accuracy), but after tuning, the gap closes considerably.

#### 4.3.4 Key Insight

The main takeaway from this notebook is that model rankings are not universal. On this small 768-record dataset, a tuned Decision Tree outperforms all the boosting methods, likely because complex ensembles need more data to generalise well. This makes a strong case for running the same kind of multi-model comparison on the full 100k-record dataset rather than assuming any single algorithm will dominate.

---
<a id='5'></a>
## 5. Techniques Not Yet Covered in Class

Going through these notebooks, I came across several methods that have not been part of our coursework so far. Here is a brief explanation of each.

### 5.1 SMOTE — Synthetic Minority Over-sampling Technique (Notebook 1)

Rather than duplicating existing minority-class samples, SMOTE generates new synthetic examples by interpolating between them in feature space. For each minority point $\mathbf{x}_i$, a new sample is created as:
$$\mathbf{x}_{\text{new}} = \mathbf{x}_i + \lambda \cdot (\mathbf{x}_{k} - \mathbf{x}_i)$$
where $\mathbf{x}_k$ is a randomly chosen k-nearest neighbour and $\lambda \sim U(0, 1)$. The result is a more diverse minority-class distribution, which is why SMOTE has become the default starting point for handling imbalanced medical datasets.

### 5.2 XGBoost (Notebooks 2 & 3)

XGBoost (Extreme Gradient Boosting) builds an additive ensemble of decision trees, fitting each new tree to the residuals of the current model. Compared to scikit-learn's `GradientBoostingClassifier`, the key improvements are: a second-order Taylor expansion of the loss during tree construction, explicit L1 and L2 regularisation terms, and parallelised column subsampling. Together these changes make XGBoost considerably faster and typically more accurate on tabular data, which is why it has been a popular Kaggle baseline for years.

### 5.3 PCA for Feature Decorrelation (Notebook 2)

PCA finds an orthogonal basis aligned with the directions of maximum variance in the feature matrix. Even when all components are retained (as in Notebook 2), the transformation still decorrelates the inputs, which can stabilise training for classifiers that are sensitive to multicollinearity. In this dataset, features like BMI and blood glucose are likely correlated since both tend to be elevated together in metabolic syndrome, so the decorrelation could be genuinely helpful.

### 5.4 GridSearchCV with Leak-Proof Pipelines (Notebooks 1 & 3)

`GridSearchCV` sweeps exhaustively over a hyperparameter grid using k-fold cross-validation. The important correctness point is that when preprocessing steps such as scaling or SMOTE are wrapped inside a `Pipeline`, they are fitted on each training fold independently rather than on the whole dataset. This prevents information from the validation fold leaking into the preprocessing step, which is a subtle but common mistake in public notebooks that apply the scaler before doing the train/test split.

### 5.5 Model Serialisation with Pickle (Notebook 2)

The author saves both trained models to disk using Python's `pickle` module, so they can be reloaded for inference later without retraining from scratch. Notebook 3 also imports `joblib`, which is generally preferred for scikit-learn models because it handles large numpy arrays more efficiently. Either approach is standard practice in any deployment-ready machine learning workflow.

---
<a id='6'></a>
## 6. My Anticipated Contributions

The existing notebooks each make valuable but narrow contributions. My mini-project will extend these works in four ways:

### 6.1 Systematic Multi-Model Comparison on the Full 100k Dataset

No existing notebook performs a rigorous multi-model comparison on the full 100,000-record dataset. I will implement a unified evaluation pipeline that trains and cross-validates the following classifiers under identical preprocessing and splitting conditions:

- Logistic Regression (interpretable baseline)
- Decision Tree
- Random Forest (tuned with GridSearchCV, as in Notebook 1)
- Gradient Boosting
- XGBoost (as in Notebook 2)
- Support Vector Machine (RBF kernel)

All models will be evaluated on the same stratified 5-fold cross-validation splits using accuracy, precision, recall, F1, and ROC-AUC. This closes the gap left by Notebook 3 (only Pima dataset) and Notebooks 1 and 2 (single-model each).

**New data preparation steps:** In addition to the duplicate removal and encoding used by prior notebooks, I will:
- Preserve the `smoking_history` recategorisation from Notebook 1 as it reduces sparsity.
- Apply `class_weight='balanced'` as an alternative to SMOTE for tree-based methods to assess its effect.
- Standardise all preprocessing inside a cross-validation pipeline to prevent leakage.

### 6.2 SHAP Explainability

Notebook 1 reports feature importance using Random Forest's built-in Gini decrease, which is known to be biased toward high-cardinality and continuous features. I will supplement this with **SHAP (SHapley Additive Explanations)** values, which provide a theoretically grounded, model-agnostic attribution of each feature's contribution to each individual prediction.

Expected outputs:
- A SHAP summary plot showing global feature importance with directionality (e.g., high HbA1c increases diabetes probability, low blood glucose decreases it)
- SHAP force plots for individual high-risk patients, explaining *why* the model flagged them, which is critical for clinical trust

### 6.3 Recall-Optimised Threshold Selection

All three existing notebooks use the default 0.5 decision threshold. In medical screening, **false negatives** (missed diabetic patients) are typically more costly than false positives. I will:

1. Plot the Precision-Recall curve for the best model.
2. Select a threshold that achieves at least **90% recall** for the diabetic class.
3. Report the precision trade-off at that threshold.

This moves evaluation from a static confusion matrix to a clinically motivated operating point, a contribution that none of the reviewed notebooks makes.

### 6.4 Stacking Ensemble

As a novel modelling contribution, I will implement a **stacking ensemble** that combines predictions from the three best single-model classifiers (e.g., Random Forest + XGBoost + Logistic Regression) using a Logistic Regression meta-learner. Stacking has not been attempted in any of the reviewed notebooks and may improve performance beyond the 96–97% accuracy ceiling observed in Notebooks 1 and 2, particularly on the minority class.

---
<a id='7'></a>
## 7. Conclusion

Looking across all three notebooks, a few consistent patterns emerge. The most important predictors are clearly `HbA1c_level` and `blood_glucose_level`: together they account for 76% of the Random Forest feature importance in Notebook 1, which makes clinical sense since these are precisely the measurements clinicians use to diagnose diabetes. Age and BMI follow at a distance, while smoking history and gender appear to add almost nothing once the blood-based markers are included.

Class imbalance at 91%/9% is a real problem that cannot be ignored. Notebooks 1 and 2 both address it in different ways — SMOTE combined with undersampling versus using XGBoost's stronger native handling of skewed distributions — and both manage to push overall accuracy into the 95–97% range. However, recall on the diabetic class remains modest, ranging from 0.67 to 0.80, which matters most in a clinical screening context where missing a diabetic patient carries real consequences.

Hyperparameter tuning made a consistent difference. Without it, Decision Trees and Random Forests overfit heavily, sometimes reaching 100% training accuracy while struggling on the test set. After tuning with `GridSearchCV`, the train/validation gap closes substantially in every case. Notebook 3 also illustrates that the best-performing model varies by dataset: on the small 768-record Pima dataset, a tuned Decision Tree outperforms XGBoost, a reminder that no single algorithm wins universally.

Building on these observations, my mini-project will contribute a systematic multi-model benchmark on the full 100k-record dataset, SHAP-based explainability to complement and critique Gini importance, recall-optimised threshold selection to make the evaluation more clinically meaningful, and a stacking ensemble as a novel modelling direction that none of the reviewed notebooks attempted.

---
<a id='8'></a>
## 8. References

**Dataset**

- Mustafa, I. (2023). *Diabetes Prediction Dataset*. Kaggle. https://www.kaggle.com/datasets/iammustafatz/diabetes-prediction-dataset

**Kaggle Notebooks Reviewed**

- @pannmie. (2023). *Diabetes EDA Random Forest HP*. Kaggle Notebook. https://www.kaggle.com/code/tumpanjawat/diabetes-eda-random-forest-hp  
  *(Local copy: `diabetes-eda-random-forest-hp.ipynb`)*

- Mubashar, M. D. (2024). *Diabetes | Hypertension Prediction (Acc 97%)*. Kaggle Notebook. https://www.kaggle.com/code/muhammaddanishmubashar/diabetes-hypertension-predict-acc-97  
  *(Local copy: `diabetes-hypertension-predict-acc-97.ipynb`)*

- Zabihullah. (2023). *Diabetes Prediction for Pima Women*. Kaggle Notebook. https://www.kaggle.com/code/zabihullah18/diabetes-prediction  
  *(Local copy: `diabetes-prediction.ipynb`)*

**Software & Libraries**

- Pedregosa, F. et al. (2011). Scikit-learn: Machine learning in Python. *JMLR*, 12, 2825–2830. https://scikit-learn.org/
- Chen, T., & Guestrin, C. (2016). XGBoost: A scalable tree boosting system. *KDD 2016*. https://doi.org/10.1145/2939672.2939785
- Chawla, N. V. et al. (2002). SMOTE: Synthetic minority over-sampling technique. *JAIR*, 16, 321–357.
- Lundberg, S. M., & Lee, S.-I. (2017). A unified approach to interpreting model predictions. *NeurIPS 2017*.
- Lemaître, G. et al. (2017). Imbalanced-learn: A Python toolbox to tackle the curse of imbalanced datasets. *JMLR*, 18(17), 1–5.